*Required packages*

In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa

*Loading dataset*

In [2]:
from google.colab import files
data_frame = files.upload()


Saving First_500 Labelled Data.xlsx to First_500 Labelled Data.xlsx


*Conversion from Excel to CSV*

In [3]:
df = pd.read_excel('First_500 Labelled Data.xlsx')

df.to_csv('Output File_First_500 Labelled Data.csv', index=False, encoding='utf-8-sig')

*CSV file download*

In [4]:
files.download('Output File_First_500 Labelled Data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

*Reading the output file of 500 annotated dataset*

In [5]:
df = pd.read_csv("Output File_First_500 Labelled Data.csv")

*Defining annotator columns*

In [6]:
annotator_columns = [
    "Shashi_Annot",
    "Tasnim_Annot",
    "Joy_Annot"
]

categories = [0, 1, 2, 3]

*Checking rows with missing annotations*

In [7]:
print("\nMissing annotation values:")
print(df[annotator_columns].isnull().sum())


#Removing rows with missing annotations
clean_df = df.dropna(subset=annotator_columns).copy()


Missing annotation values:
Shashi_Annot    0
Tasnim_Annot    0
Joy_Annot       0
dtype: int64


*All labels are counted as integer*

In [8]:
for col in annotator_columns:
    clean_df[col] = clean_df[col].astype(int)

*Checking if any invalid label is given as input*

In [9]:
invalid_labels = {}

for col in annotator_columns:
    invalid = clean_df[~clean_df[col].isin(categories)][col].unique()

    if len(invalid) > 0:
        invalid_labels[col] = invalid

if invalid_labels:
    print("Invalid labels have been found: ")
    print(invalid_labels)
else:
    print("No invalid label has been found.")

No invalid label has been found.


***Implementing Fleiss Kappa***

In [10]:
rating_matrix = []

for _, row in clean_df.iterrows():

    labels = row[annotator_columns].tolist()

    counts = [labels.count(category)
        for category in categories]

    rating_matrix.append(counts)


rating_matrix = np.array(rating_matrix)

*Calculation of Fleiss Kappa*

In [11]:
kappa = fleiss_kappa(
    rating_matrix,
    method="fleiss"
)

print("Fleiss Kappa result")

print(f"The Fleiss's Kappa is: {kappa:.4f}")

Fleiss Kappa result
The Fleiss's Kappa is: 0.5968


*Calculating percentage of complete agreement*

In [12]:
def complete_agreement(row):

    labels = row[annotator_columns].tolist()

    return len(set(labels)) == 1


clean_df["complete_agreement"] = clean_df.apply(
    complete_agreement,
    axis=1
)


complete_agreement_count = clean_df[
    "complete_agreement"].sum()

total_items = len(clean_df)

agreement_percentage = (complete_agreement_count / total_items) * 100


print(f"Complete agreement: "f"{complete_agreement_count}/{total_items}")

print(f"Percentage agreement: "f"{agreement_percentage:.2f}%")

Complete agreement: 293/500
Percentage agreement: 58.60%


*Identifying disagreement cases*

In [14]:
disagreement_df = clean_df[clean_df["complete_agreement"] == False].copy()

print(f"The total number of disagreement cases is: "f"{len(disagreement_df)}")

The total number of disagreement cases is: 207


*Calculating majority vote label*

In [15]:
def Majority_Vote(row):

    labels = row[annotator_columns].tolist()

    counts = {}

    for label in labels:
        counts[label] = counts.get(label, 0) + 1

    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    #When there is a clear majority
    if len(winners) == 1:
        return winners[0]

    #3 annotators disagree
    return np.nan


clean_df["Majority_Vote"] = clean_df.apply(
    Majority_Vote,
    axis=1
)


*Identifying disagreements where all of them disagree*

In [16]:
def Three_way_Disagreement(row):

    labels = row[annotator_columns].tolist()

    return len(set(labels)) == 3


clean_df["Three_way_Disagreement"] = clean_df.apply(
    Three_way_Disagreement,
    axis=1
)


three_way_count = clean_df["Three_way_Disagreement"].sum()


print("Three-way disagreements:")
print(three_way_count)

Three-way disagreements:
5


*Showing three-way disagreement data*

In [17]:
columns_to_display = [
    "Serial No",
    "Shashi_Annot",
    "Tasnim_Annot",
    "Joy_Annot",
    "Majority_Vote",
    "Three_way_Disagreement"
]

print(clean_df[clean_df["complete_agreement"] == False][columns_to_display])

     Serial No  Shashi_Annot  Tasnim_Annot  Joy_Annot  Majority_Vote  \
5            6             3             3          2            3.0   
9           10             2             3          2            2.0   
11          12             1             1          0            1.0   
12          13             1             0          0            0.0   
15          16             1             2          2            2.0   
..         ...           ...           ...        ...            ...   
488        489             3             2          2            2.0   
490        491             1             0          0            0.0   
494        495             3             2          3            3.0   
497        498             3             2          1            NaN   
498        499             1             0          0            0.0   

     Three_way_Disagreement  
5                     False  
9                     False  
11                    False  
12             

*Saving results*

In [18]:
clean_df.to_csv("Annotation_Results.csv",index=False)

#Save disagreement cases
disagreement_df.to_csv("Disagreement_Cases.csv",index=False)

print("Annotation Results and Disagreement Cases have been saved successfully.")

Annotation Results and Disagreement Cases have been saved successfully.


In [19]:
clean_df.to_csv(
    "Annotation_Results.csv",
    index=False,
    encoding='utf-8-sig'
)

disagreement_df.to_csv(
    "Disagreement_Cases.csv",
    index=False,
    encoding='utf-8-sig'
)

*Downloading both files*

In [20]:
files.download('Annotation_Results.csv')
files.download('Disagreement_Cases.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>